# Task Arithmetic: Negation, Addition, and Analogy — Colab

This notebook executes the repository's existing Python pipeline using **T5-small**. It does not contain separate implementations or sample experimental results.

**Requirements:** A GPU runtime is recommended (`Runtime → Change runtime type → GPU`). The full run performs four independent full-parameter fine-tunes and evaluates three arithmetic operations. Results depend on the runtime and training setup.

**Getting started:** Download a release ZIP of this repository, or clone it into `/content/task-arithmetic-minimal-reproduction`. When prompted below, upload the repository ZIP.


## 1. Load repository

If the repository is already present in `/content`, it is used directly. Otherwise upload the ZIP and the notebook will extract it. Only the repository contents are needed; no tokens or credentials are required for public datasets.

In [ ]:
from pathlib import Path
import os, zipfile

ROOT = Path('/content/task-arithmetic-minimal-reproduction')
if not (ROOT / 'scripts/run_all.sh').exists():
    from google.colab import files
    print('Upload the project ZIP file:')
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
    if not zip_names:
        raise ValueError('Please upload a repository ZIP.')
    with zipfile.ZipFile(zip_names[0]) as archive:
        archive.extractall('/content')

if not (ROOT / 'scripts/run_all.sh').exists():
    candidates = list(Path('/content').glob('*/scripts/run_all.sh'))
    if not candidates:
        raise FileNotFoundError('No scripts/run_all.sh found in the extracted repository.')
    ROOT = candidates[0].parent.parent

os.chdir(ROOT)
print('Repository:', ROOT)


## 2. Check environment and install dependencies

An available CUDA GPU reduces the runtime significantly. Installation installs the packages declared by the repository.

In [ ]:
import subprocess, sys

def run_module(*args):
    subprocess.run([sys.executable, '-m', *args], check=True, cwd=ROOT)

run_module('pip', 'install', '-r', 'requirements.txt')
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


## 3. Test the arithmetic and data utilities

These tests check local logic; passing them does **not** establish empirical effectiveness.

In [ ]:
run_module('pytest', '-q')


## 4. Prepare public datasets

Real (not synthetic) data are streamed from Hugging Face. The Yelp sentiment *test* labels are never used for fine-tuning or hyperparameter selection.

In [ ]:
run_module('scripts.prepare_data', '--mode', 'real', '--train', '1600', '--val', '200', '--test', '250', '--lm-train', '2500', '--lm-val', '300')


## 5. Fine-tune four independent checkpoints

Each task starts from the same pretrained T5-small weights. The LM tasks use a reduced proxy objective; their results should not be described as an exact reproduction of the original paper.

In [ ]:
TASKS = ('amazon_sent', 'agnews_topic', 'amazon_lm', 'yelp_lm')
for task in TASKS:
    print(f'\n===== Training {task} =====', flush=True)
    run_module('scripts.train_task', '--task', task, '--max-steps', '180')


## 6. Extract Task Vectors and verify reconstruction

Vectors are full-model weight deltas from the same initialization. The script checks that adding each vector back to the base at coefficient 1 reconstructs the corresponding fine-tuned checkpoint.

In [ ]:
run_module('scripts.extract_vectors')


## 7. Evaluate Negation, Addition, and Analogy

The evaluation uses real held-out data. Analogy performance on Yelp is measured on the held-out test split; only the prespecified domain coefficient is a confirmatory comparison.

In [ ]:
run_module('scripts.run_experiments', '--val-size', '80', '--test-size', '250')


## 8. Generate and inspect the measured plots

Graphs are generated from the measured `results/run_summary.json`, not hard-coded demonstration values.

In [ ]:
run_module('scripts.plot_results')
from IPython.display import display, Image
for name in ('negation','addition','analogy'):
    path = ROOT / 'results' / 'figures' / f'{name}.png'
    if not path.is_file():
        raise FileNotFoundError(path)
    display(Image(filename=str(path)))


## 9. Export lightweight results

Export only metrics and figures. Trained checkpoints and downloaded datasets are large and are excluded. Preserve the files before the ephemeral Colab session ends.

**Interpretation:** Accuracy improvements are not guaranteed; inspect the control-task changes and the limitations described in `README.md`.

In [ ]:
from google.colab import files
results = [ROOT/'results'/'run_summary.json', ROOT/'results'/'metrics.csv']
results += sorted((ROOT/'results'/'figures').glob('*.png'))
missing = [p for p in results if not p.is_file()]
if missing:
    raise FileNotFoundError(f'Missing result files: {missing}')
output_zip = Path('/content/task_arithmetic_results.zip')
with zipfile.ZipFile(output_zip, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in results:
        archive.write(path, path.relative_to(ROOT))
print('Saved:', output_zip)
files.download(str(output_zip))
